<a href="https://colab.research.google.com/github/mohammadabuhamed2/binx_mohammadabuhamed/blob/main/day5_week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM vs Transformer for Sentiment Analysis

## Step 1 — Confirm and Justify the Core Architecture

The project uses text data for binary sentiment classification on the IMDB movie review dataset.

A Transformer-based architecture was selected as the improved core architecture because Transformers are well suited for understanding contextual relationships between words in long text sequences.

RoBERTa was selected because it is a pretrained encoder-based Transformer designed for language understanding tasks such as text classification.

Using a pretrained RoBERTa model also allows transfer learning, where previously learned language representations are adapted to the IMDB sentiment classification task through fine-tuning.

## Load the IMDB Dataset

### Download the Dataset from Kaggle

In [1]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print(path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
/kaggle/input/imdb-dataset-of-50k-movie-reviews


### Check Dataset Files

In [2]:
import os
print(os.listdir(path))

['IMDB Dataset.csv']


### Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

### Load the Dataset

In [3]:
df = pd.read_csv(path + "/IMDB Dataset.csv")

### Preview the Dataset

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### Check Dataset Shape

In [5]:
print("Dataset shape:", df.shape)

Dataset shape: (50000, 2)


### Check Dataset Information

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


## Explore Sentiment Distribution

### Check Sentiment Counts

In [7]:
print(df["sentiment"].value_counts(normalize=True)*100)

sentiment
positive    50.0
negative    50.0
Name: proportion, dtype: float64


## Prepare the Target Labels

In [8]:
df["sentiment"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

### Check Encoded Labels

In [9]:
print(df["sentiment"].value_counts())

sentiment
1    25000
0    25000
Name: count, dtype: int64


## Split Features and Target

In [10]:
X = df["review"]
y = df["sentiment"]

### Split the Dataset

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

### Check Dataset Splits

In [12]:
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (40000,) (40000,)
Validation: (5000,) (5000,)
Test: (5000,) (5000,)


## Prepare Text for the LSTM

### Create the Text Vectorization Layer

In [13]:
text_victorizer=tf.keras.layers.TextVectorization(max_tokens=10000,output_mode='int',output_sequence_length=400)

### Adapt the Vectorizer to the Training Text

In [14]:
text_victorizer.adapt(X_train)

## Build the LSTM Sentiment Model

In [51]:
lstm_model = tf.keras.Sequential([
    tf.keras.Input(shape=(400,)),
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=128,
        mask_zero=True
    ),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

### Display the Model Architecture

In [52]:
lstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 400, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

### Compile the LSTM Model

In [53]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

### Configure Early Stopping

In [54]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

### Configure Model Checkpoint

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [35]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/binx_training/day5_week7/best_lstm_sentiment_model.keras",
    monitor="val_loss",
    save_best_only=True
)

### Train the LSTM Model

####vectorize the data

In [15]:
X_train_vec = text_victorizer(X_train)
X_val_vec = text_victorizer(X_val)
X_test_vec = text_victorizer(X_test)

In [56]:
history = lstm_model.fit(
    X_train_vec,
    y_train,
    validation_data=(X_val_vec, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stopping, checkpoint]
)

Epoch 1/200
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 546s 434ms/step - accuracy: 0.7990 - loss: 0.4425 - val_accuracy: 0.8698 - val_loss: 0.3268
Epoch 2/200
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 569s 440ms/step - accuracy: 0.8432 - loss: 0.3552 - val_accuracy: 0.7448 - val_loss: 0.5165
Epoch 3/200
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 510s 398ms/step - accuracy: 0.9032 - loss: 0.2399 - val_accuracy: 0.8982 - val_loss: 0.2568
Epoch 4/200
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 521s 416ms/step - accuracy: 0.9434 - loss: 0.1565 - val_accuracy: 0.8970 - val_loss: 0.2542
Epoch 5/200
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 527s 389ms/step - accuracy: 0.9636 - loss: 0.1069 - val_accuracy: 0.8918 - val_loss: 0.3077
Epoch 6/200
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 602s 469ms/step - accuracy: 0.9785 - loss: 0.0697 - val_accuracy: 0.8918 - val_loss: 0.3517
Epoch 7/200
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 514s 411ms/step - accuracy: 0.9871 - loss: 0.0438 - val_accuracy: 0.8892 - val_loss: 0.4317
Epoch 8/200
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 542s 434ms/s

### Load the Saved LSTM Model

In [19]:
new_model=tf.keras.models.load_model("/content/drive/MyDrive/binx_training/day5_week7/best_lstm_sentiment_model.keras")

### Evaluate the LSTM Model

In [20]:
test_loss,test_accuracy=new_model.evaluate(X_test_vec,y_test)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 45ms/step - accuracy: 0.8976 - loss: 0.2710
Test Loss: 0.2709592580795288
Test Accuracy: 0.897599995136261


### LSTM Test Results

The LSTM sentiment classification model achieved strong performance on the IMDB test dataset.

- **Test Accuracy:** 89.76%
- **Test Loss:** 0.2710

The model was able to learn sequential patterns in the movie reviews and classify most reviews correctly as positive or negative.

### Generate LSTM Predictions

In [21]:
y_pred_prob =new_model.predict(X_test_vec)

y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step


### Classification Report

In [23]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.91      0.90      2500
           1       0.90      0.89      0.90      2500

    accuracy                           0.90      5000
   macro avg       0.90      0.90      0.90      5000
weighted avg       0.90      0.90      0.90      5000



### Classification Report Analysis

The LSTM model achieved balanced performance across both sentiment classes.

- Class 0 (Negative) achieved an F1-score of 0.90.
- Class 1 (Positive) also achieved an F1-score of 0.90.
- The overall accuracy was approximately 90%.
- Precision and recall were similar for both classes, indicating that the model performs consistently on both positive and negative reviews.

Overall, the LSTM model showed good generalization on the test dataset.

## Transformer Sentiment Analysis

A pretrained Transformer model from Hugging Face is used to classify the IMDB movie reviews.

The same test dataset used by the LSTM model will be used to compare both models fairly.

In [22]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

### Generate Transformer Predictions

In [23]:
transformer_results = classifier(
    X_test.tolist(),
    batch_size=32,
    truncation=True
)

### Convert Transformer Predictions to Numeric Labels

In [24]:
transformer_pred = []
for result in transformer_results:
    if result["label"] == "POSITIVE":
        transformer_pred.append(1)
    else:
        transformer_pred.append(0)

### Transformer Classification Report

In [25]:
from sklearn.metrics import classification_report
print(classification_report(y_test,transformer_pred))

              precision    recall  f1-score   support

           0       0.86      0.93      0.89      2500
           1       0.92      0.85      0.89      2500

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.89      0.89      0.89      5000



### Transformer Classification Report Analysis

The pretrained Transformer achieved approximately 89% accuracy on the IMDB test dataset.

- Class 0 (Negative) achieved an F1-score of 0.89.
- Class 1 (Positive) achieved an F1-score of 0.89.
- The model showed higher recall for negative reviews than positive reviews.
- Overall performance was strong, but slightly lower than the LSTM model on this test set.

## Step 2 — Train and Tune the Improved Core Model

The pretrained RoBERTa model is prepared and fine-tuned on the IMDB training dataset.

The workflow includes:

- Preparing TensorFlow datasets and batches.
- Tokenizing the movie reviews using the matching RoBERTa tokenizer.
- Applying padding, truncation, and special tokens.
- Creating the `token_ids` and `padding_mask` inputs required by RoBERTa.
- Adding a task-specific binary classification head.
- Compiling the model with Adam and a small learning rate.
- Using EarlyStopping and ModelCheckpoint during fine-tuning.
- Evaluating the best saved model on the test dataset.

## Fine-Tuning a Pretrained RoBERTa Transformer

In this section, a pretrained RoBERTa Transformer will be fine-tuned on the IMDB movie review dataset for binary sentiment classification.

The pretrained model already contains language representations learned during pretraining. Fine-tuning will adapt these pretrained weights to our specific task of classifying movie reviews as negative or positive.

The same training, validation, and test splits used with the LSTM model will be reused to allow a fair comparison between the two architectures.

In [14]:
import tensorflow as tf
import keras
from keras import layers
import keras_hub

### Prepare TensorFlow Datasets

The existing Pandas training, validation, and test splits are converted into `tf.data.Dataset` objects.

Using TensorFlow datasets allows the text and labels to be processed efficiently in batches during training.

- The training dataset is shuffled before training.
- The training, validation, and test datasets are divided into batches.
- Validation and test data are not shuffled because they are only used for evaluation.

### batch size

In [15]:
batch_size = 16

### Create the Training TensorFlow Dataset

The training reviews and their corresponding sentiment labels are combined into a TensorFlow `Dataset`.

Each movie review is paired with its correct target label:

- `0` represents a negative review.
- `1` represents a positive review.

The reviews are converted to strings, while the labels are converted to a NumPy array before creating the TensorFlow dataset.

In [16]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train.astype(str).tolist(), y_train.to_numpy()))
val_ds = tf.data.Dataset.from_tensor_slices((X_val.astype(str).tolist(), y_val.to_numpy()))
test_ds = tf.data.Dataset.from_tensor_slices((X_test.astype(str).tolist(), y_test.to_numpy()))

### Batch the TensorFlow Datasets

The datasets are divided into smaller batches so that the model processes only a limited number of movie reviews at a time.

A batch size of 16 is used to reduce GPU memory usage while fine-tuning the large pretrained RoBERTa model.

In [17]:
train_ds = train_ds.shuffle(10000,seed=42).batch(batch_size)
val_ds = val_ds.batch(batch_size)
test_ds = test_ds.batch(batch_size)

### Load the Pretrained RoBERTa Tokenizer and Backbone

The pretrained `roberta_base_en` preset is loaded from KerasHub.

Two matching components are required:

- **Tokenizer:** Converts raw movie reviews into token IDs that RoBERTa understands.
- **Backbone:** Contains the pretrained RoBERTa Transformer architecture and its learned weights.

The tokenizer and backbone must come from the same preset so that the token IDs produced by the tokenizer match the vocabulary and representations learned by the pretrained model.

In [18]:
roberta_tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
roberta_backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

100%|██████████| 445/445 [00:00<00:00, 1.19MB/s]


100%|██████████| 686/686 [00:00<00:00, 1.03MB/s]


100%|██████████| 0.99M/0.99M [00:00<00:00, 3.20MB/s]


100%|██████████| 446k/446k [00:00<00:00, 1.79MB/s]


100%|██████████| 474M/474M [00:09<00:00, 50.6MB/s]


### Inspect the Loaded RoBERTa Components

The loaded objects are inspected to verify that KerasHub selected the correct RoBERTa tokenizer and pretrained backbone from the preset.

In [19]:
print(type(roberta_tokenizer))
print(type(roberta_backbone))

<class 'keras_hub.src.models.roberta.roberta_tokenizer.RobertaTokenizer'>
<class 'keras_hub.src.models.roberta.roberta_backbone.RobertaBackbone'>


### Inspect the Pretrained RoBERTa Architecture

The model summary is displayed to inspect the architecture and number of parameters in the pretrained RoBERTa backbone before adding the task-specific classification head.

In [16]:
roberta_backbone.summary()

Model: "roberta_backbone"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings          │ (None, None, 768) │ 38,996,736 │ token_ids[0][0]   │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_layer_n… │ (None, None, 768) │      1,536 │ embeddings[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_dropout  │ (None, None, 768) │          0 │ embeddings_layer… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ padding_mask        │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_0 │ (None, None, 768) │  7,087,872 │ embeddings_dropo… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_1 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_2 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_3 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_4 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_5 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_6 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_7 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_8 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_9 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_… │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_… │ (None, None, 768) │  7,087,872 │ transformer_laye

 Total params: 124,052,736 (473.22 MB)

 Trainable params: 124,052,736 (473.22 MB)

 Non-trainable params: 0 (0.00 B)

### RoBERTa Backbone Architecture

The pretrained RoBERTa Base backbone contains:

- Token and positional embedding layers.
- A padding mask input.
- 12 Transformer Encoder layers.
- A hidden representation size of 768.
- Approximately 124 million trainable parameters.

The backbone is pretrained, meaning its parameters already contain knowledge learned during language pretraining. During fine-tuning, these weights can be updated to adapt RoBERTa to the IMDB sentiment classification task.

At this stage, the backbone does not yet contain the final binary classification head for predicting positive or negative sentiment.

### Prepare RoBERTa Sequences with StartEndPacker

RoBERTa expects tokenized text to follow the same sequence format used during pretraining.

Each sequence should:

- Start with a special start token `<s>`.
- End with a special end token `</s>`.
- Be padded with `<pad>` tokens when it is shorter than the selected sequence length.
- Be truncated when it is longer than the selected sequence length.

KerasHub's `StartEndPacker` performs these operations and also creates a padding mask that tells RoBERTa which positions contain real tokens and which positions contain padding.

In [20]:
sequence_length = 256

### Set the RoBERTa Sequence Length

A sequence length of 512 tokens is used for the RoBERTa model.

This means that each movie review will be represented using a maximum of 512 token positions.

- Reviews shorter than 512 tokens will be padded.
- Reviews longer than 512 tokens will be truncated.
- Using 512 tokens allows the model to preserve more information from long movie reviews.

### Create the RoBERTa Start-End Packer

The `StartEndPacker` prepares tokenized reviews in the format expected by the pretrained RoBERTa model.

It uses the special start, end, and padding token IDs defined by the matching RoBERTa tokenizer.

The packer also returns a padding mask that will be passed to the RoBERTa backbone together with the token IDs.

In [21]:
packer = keras_hub.layers.StartEndPacker(
    sequence_length=sequence_length,
    start_value=roberta_tokenizer.start_token_id,
    end_value=roberta_tokenizer.end_token_id,
    pad_value=roberta_tokenizer.pad_token_id,
    return_padding_mask=True
)

### Create the RoBERTa Preprocessing Function

A preprocessing function is created to transform each batch of raw movie reviews into the input format expected by the pretrained RoBERTa backbone.

The function performs three main operations:

- It tokenizes the raw text using the matching RoBERTa tokenizer.
- It packs the token IDs into fixed-length sequences of 512 positions using the `StartEndPacker`.
- It creates a `padding_mask` so that RoBERTa can distinguish real tokens from padding.

The function returns the model inputs as a dictionary containing `token_ids` and `padding_mask`, while the original sentiment labels are returned unchanged.

In [22]:
def preprocess(text,label):
  token_ids=roberta_tokenizer(text)
  token_ids,padding_mask=packer(token_ids)
  return {
      "token_ids":token_ids,
      "padding_mask":padding_mask
  },label

### Apply RoBERTa Preprocessing to the Datasets

The preprocessing function is applied to every batch in the training, validation, and test datasets using the TensorFlow `map()` operation.

After this step, the raw movie review text is replaced by the `token_ids` and `padding_mask` inputs required by the RoBERTa backbone, while the sentiment labels remain unchanged.

In [23]:
preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

### Build the RoBERTa Classification Head

The pretrained RoBERTa backbone produces a contextual representation for every token in the input sequence.

Its output has the shape:

`(batch_size, sequence_length, 768)`

For binary sentiment classification, we need one prediction for each movie review rather than one representation for every token.

The representation of the first token is therefore selected as a summary representation of the full sequence. Because this token can attend to all other tokens through self-attention, its final representation contains contextual information from the complete review.

A classification head is then added using:

- Dropout for regularization.
- A Dense layer with ReLU activation for further feature transformation.
- A final Dense layer with sigmoid activation to produce the probability of the positive class.

In [19]:
inputs = roberta_backbone.input
x = roberta_backbone(inputs)
x = x[:, 0, :]
x = layers.Dropout(0.1)(x)
x = layers.Dense(768,activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1,activation="sigmoid")(x)
classifier = keras.Model(inputs,outputs)

### Inspect the RoBERTa Sentiment Classifier

The complete classifier now combines the pretrained RoBERTa backbone with a task-specific binary classification head.

The model summary is displayed to inspect the complete architecture before fine-tuning.

In [40]:
classifier.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ padding_mask        │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ roberta_backbone    │ (None, None, 768) │ 124,052,7… │ padding_mask[0][… │
│ (RobertaBackbone)   │                   │            │ token_ids[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 768)       │          0 │ roberta_backbone… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 768)       │          0 │ get_item_1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 768)       │    590,592 │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 768)       │          0 │ dense_2[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │        769 │ dropout_15[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 124,644,097 (475.48 MB)

 Trainable params: 124,644,097 (475.48 MB)

 Non-trainable params: 0 (0.00 B)

### RoBERTa Sentiment Classifier Architecture

The final sentiment classifier combines the pretrained RoBERTa backbone with a custom binary classification head.

The architecture contains:

- Two inputs: `token_ids` and `padding_mask`.
- The pretrained RoBERTa backbone for contextual text representation.
- Selection of the first token representation.
- Dropout for regularization.
- A Dense layer with 768 neurons and ReLU activation.
- A final Dense layer with one neuron and sigmoid activation for binary sentiment classification.

All RoBERTa parameters are currently trainable, allowing the pretrained weights to be updated during fine-tuning.

### Compile the RoBERTa Classifier

The RoBERTa sentiment classifier is compiled before fine-tuning.

A small learning rate is used because RoBERTa already contains pretrained weights. Fine-tuning should update these weights gradually instead of making large changes that could destroy the useful language representations learned during pretraining.

Binary cross-entropy is used because the task has two classes: negative and positive.

Accuracy is used to monitor the percentage of correctly classified movie reviews.

In [20]:
classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

### Add Early Stopping and Model Checkpoint

Callbacks are used to make fine-tuning safer and more efficient.

- `EarlyStopping` stops training when the validation loss stops improving and restores the best model weights.
- `ModelCheckpoint` saves the best model during training based on validation loss.

These callbacks help reduce unnecessary training, limit overfitting, and preserve the best version of the fine-tuned RoBERTa model.

In [42]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [43]:
checkpoint = keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/binx_training/day5_week7/best_roberta_model.keras",
    monitor="val_loss",
    save_best_only=True
)

### Fine-Tune the RoBERTa Classifier

The pretrained RoBERTa classifier is fine-tuned on the IMDB training dataset.

During fine-tuning, the pretrained RoBERTa weights and the newly added classification head are updated together using the sentiment labels.

The validation dataset is used to monitor how well the model generalizes to unseen reviews during training.

In [ ]:
history = classifier.fit(
    preprocessed_train_ds,
    validation_data=preprocessed_val_ds,
    epochs=10,
    callbacks=[early_stopping, checkpoint]
)

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2125s 822ms/step - accuracy: 0.9060 - loss: 0.2447 - val_accuracy: 0.9276 - val_loss: 0.1807
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2036s 814ms/step - accuracy: 0.9359 - loss: 0.1735 - val_accuracy: 0.9308 - val_loss: 0.1810
Epoch 3/10
2195/2500 ━━━━━━━━━━━━━━━━━━━━ 3:57 778ms/step - accuracy: 0.9518 - loss: 0.1336

### Load the Best Saved RoBERTa Model

The best RoBERTa sentiment classifier saved during fine-tuning is loaded from Google Drive.

Because `ModelCheckpoint` was configured with `save_best_only=True`, this file contains the model version that achieved the best validation loss during training.

In [24]:
classifier = keras.models.load_model(
    "/content/drive/MyDrive/binx_training/day5_week7/best_roberta_model.keras")

### Evaluate the Fine-Tuned RoBERTa Model

The best saved RoBERTa model is evaluated on the test dataset.

The test dataset was not used during training or validation, so this evaluation measures how well the fine-tuned model generalizes to unseen movie reviews.

The final test loss and test accuracy are reported.

In [25]:
test_loss, test_accuracy = classifier.evaluate(
    preprocessed_test_ds
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 102s 296ms/step - accuracy: 0.9258 - loss: 0.1886
Test Loss: 0.18859313428401947
Test Accuracy: 0.9258000254631042


### Fine-Tuned RoBERTa Test Results

The fine-tuned RoBERTa model achieved strong performance on the IMDB test dataset.

- **Test Accuracy:** 92.58%
- **Test Loss:** 0.1886

The fine-tuned RoBERTa model outperformed both the LSTM baseline and the pretrained Transformer used without task-specific fine-tuning.

This demonstrates the benefit of adapting a pretrained Transformer to the target dataset through fine-tuning.

### Generate Predictions with the Fine-Tuned RoBERTa Model

The fine-tuned RoBERTa model is used to generate sentiment predictions for the test dataset.

Because the final output layer uses a sigmoid activation, the model produces one probability between 0 and 1 for each movie review.

A probability of 0.5 or higher is classified as positive, while a probability below 0.5 is classified as negative.

In [26]:
roberta_pred_prob = classifier.predict(preprocessed_test_ds)

313/313 ━━━━━━━━━━━━━━━━━━━━ 109s 321ms/step


### Convert Probabilities into Class Labels

The sigmoid probabilities are converted into final binary class predictions using a threshold of 0.5.

- Values below 0.5 are classified as `0` (Negative).
- Values greater than or equal to 0.5 are classified as `1` (Positive).

In [28]:
roberta_pred = (roberta_pred_prob >= 0.5).astype(int).flatten()

### RoBERTa Classification Report

A classification report is generated to evaluate the fine-tuned RoBERTa model in more detail.

The report includes:

- Precision
- Recall
- F1-score
- Support

for both negative and positive movie reviews.

In [29]:
from sklearn.metrics import classification_report
print(classification_report(y_test,roberta_pred))

              precision    recall  f1-score   support

           0       0.92      0.93      0.93      2500
           1       0.93      0.92      0.93      2500

    accuracy                           0.93      5000
   macro avg       0.93      0.93      0.93      5000
weighted avg       0.93      0.93      0.93      5000



### RoBERTa Classification Report Analysis

The fine-tuned RoBERTa model achieved balanced and strong performance across both sentiment classes.

- Class `0` (Negative) achieved an F1-score of `0.93`.
- Class `1` (Positive) also achieved an F1-score of `0.93`.
- Precision and recall were very similar for both classes.
- The overall accuracy was approximately `93%`.
- The balanced results indicate that the model performs consistently on both positive and negative movie reviews.

Overall, fine-tuned RoBERTa generalized well to the unseen IMDB test dataset.

## Step 3 — Compare Model Performance

The Sprint 2 model is compared with the previous models using the same IMDB test dataset.

The comparison focuses on test accuracy, F1-score, and test loss to determine whether the improved architecture provided better generalization.

| Model | Test Accuracy | F1-Score | Test Loss | Training Approach |
|---|---:|---:|---:|---|
| LSTM | 89.76% | 0.90 | 0.2710 | Trained on IMDB |
| Pretrained Transformer | ~89% | 0.89 | N/A | Used without fine-tuning |
| Fine-Tuned RoBERTa | 92.58% | 0.93 | 0.1886 | Fine-tuned on IMDB |

The fine-tuned RoBERTa model achieved the best overall performance.

Fine-tuning improved the Transformer by adapting its pretrained language representations to the specific IMDB sentiment classification task.

## Step 5 — Sprint Review and Retrospective

### Sprint Review

The goal of Sprint 2 was to improve the text classification model by using a stronger architecture and applying transfer learning.

During this sprint:

- An LSTM model was used as a sequence-based baseline for sentiment classification.
- A pretrained Transformer was tested without fine-tuning.
- A pretrained RoBERTa Transformer was fine-tuned on the IMDB sentiment dataset.
- EarlyStopping and ModelCheckpoint were used to control training and preserve the best model.
- The final models were evaluated using accuracy, loss, precision, recall, and F1-score.

### Final Results

| Model | Test Accuracy | F1-Score | Test Loss |
|---|---:|---:|---:|
| LSTM | 89.76% | 0.90 | 0.2710 |
| Pretrained Transformer | ~89% | 0.89 | N/A |
| Fine-Tuned RoBERTa | 92.58% | 0.93 | 0.1886 |

The fine-tuned RoBERTa model achieved the strongest overall performance.

Fine-tuning allowed the pretrained Transformer to adapt its existing language knowledge to the specific IMDB sentiment classification task.

### Retrospective

#### What Went Well

- The LSTM provided a strong baseline for comparison.
- Transfer learning with RoBERTa improved the final classification performance.
- EarlyStopping and ModelCheckpoint helped protect the training process.
- The final RoBERTa model achieved balanced precision, recall, and F1-score across both sentiment classes.

#### What Could Be Improved

The initial RoBERTa experiment used a sequence length of 512 tokens, which made training very slow and computationally expensive.

Reducing the sequence length to 256 significantly improved training efficiency while still preserving strong model performance.

#### Concrete Change for Sprint 3

In Sprint 3, computational cost will be tested on a small training run before starting full model training.

Sequence length, batch size, and estimated training time will be checked first to avoid unnecessary long training runs.